In [6]:
import sys
print(sys.executable)

import torch

print(torch.__version__)
print("CUDA available:", torch.cuda.is_available())

/m/home/home6/69/taivait1/unix/dippa/ceremegbellum/venv/bin/python
2.10.0+cu128
CUDA available: True


In [7]:
import os.path as op
from pathlib import Path
import mne
import pickle
import numpy as np
import matplotlib.pyplot as plt
from mne.datasets import sample
from cmb import get_cerebellum_data, setup_full_source_space, plot_cerebellum_data, CMB_DATA_DIR

In [8]:
data_path = sample.data_path()

# Paths to subject data
sample_dir = op.join(data_path, 'MEG', 'sample',)
raw_fname = op.join(sample_dir, 'sample_audvis_raw.fif')
subjects_dir = op.join(data_path, 'subjects')
subject = 'sample'
trans = op.join(sample_dir, 'sample_audvis_raw-trans.fif')
fname_cov = sample_dir + '/sample_audvis-cov.fif'
evo_fname = sample_dir + '/sample_audvis-ave.fif'

# Check if the required data are available and download if not
get_cerebellum_data()

The required atlas data and segmentation models seem to be downloaded.


In [9]:
# Cerebellar specific
cb_data = pickle.load(open(op.join(CMB_DATA_DIR, 'data', 'cerebellum_geo'), 'rb'))
spacing = 2 # Use spacing 2 to get an approximately equal grid density in cerebral and cerebellar cortices

In [ ]:
# Setup source space
cerebellum_subsampling = 'dense'
src_whole = setup_full_source_space(subject, subjects_dir, cerb_subsampling=cerebellum_subsampling,
                                    plot_cerebellum=False, spacing=spacing)

Setting up the source space with the following parameters:

SUBJECTS_DIR = /u/69/taivait1/unix/mne_data/MNE-sample-data/subjects
Subject      = sample
Surface      = white
Approximate spacing 2 mm

>>> 1. Creating the source space...

Loading /u/69/taivait1/unix/mne_data/MNE-sample-data/subjects/sample/surf/lh.white...
Mapping lh sample -> spacing (2) ...
    Triangle neighbors and vertex normals...
    Vertex neighbors...
    Decimating...
loaded lh.white 84259/155407 selected to source space (spacing = 2)

Loading /u/69/taivait1/unix/mne_data/MNE-sample-data/subjects/sample/surf/rh.white...
Mapping rh sample -> spacing (2) ...
    Triangle neighbors and vertex normals...
    Vertex neighbors...
    Decimating...
loaded rh.white 85236/156866 selected to source space (spacing = 2)

You are now one step closer to computing the gain matrix
starting subject sample...
Doing segmentation...


Please cite the following paper when using nnUNet:

Isensee, F., Jaeger, P.F., Kohl, S.A.A. et al. 

In [ ]:
save_path = Path.cwd()
print(save_path)

In [ ]:
source_space_save_path = save_path / "src_whole.fif")
src_whole.save(source_space_save_path)

In [ ]:
# Compute forward and inverse operators
conductivity=(0.3, 0.006, 0.3)
mindist = 3.0 # important not to use too large mindist because the cerebellar cortex and inner skull boundary are usually within 5 mm
model = mne.make_bem_model(subject=subject, ico=4, conductivity=conductivity, subjects_dir=subjects_dir)
bem = mne.make_bem_solution(model) # IF too many source space points are lost, the inner skull boundary is too tight and need to be expanded.
info = mne.io.read_info(raw_fname)
fwd = mne.make_forward_solution(info, trans, src_whole, bem=bem, mindist=mindist, eeg=True, n_jobs=1)
fwd = mne.convert_forward_solution(fwd, surf_ori=True, force_fixed=True, copy=True)
noise_cov = mne.read_cov(fname_cov)
inverse_operator = mne.minimum_norm.make_inverse_operator(info, fwd, noise_cov, depth=None, fixed=True)

In [ ]:
# Example forward simulation from patch in right lobule VIIIa
label =  cb_data['parcellation']['fine labels'][714]
active_verts = np.where(np.isin(cb_data['dw_data'][cerebellum_subsampling], label.vertices))[0]
active_verts = np.where(np.isin(fwd['src'][1]['vertno'], active_verts))[0]
act_cerb = np.zeros((fwd['src'][1]['nuse']))
act_cerb[active_verts] = 1

# Plot activated patch (set view to 'all' if not remote connecting)
plot_cerebellum_data(act_cerb, fwd['src'], src_whole, cb_data, cort_data=np.zeros(fwd['src'][0]['nuse']), flatmap_cmap='bwr', mayavi_cmap='OrRd',
                     smoothing_steps=0, view='flatmap', sub_sampling=cerebellum_subsampling, cmap_lims=[0,100])

In [ ]:
# Plot estimated activation (set view to 'all' if not remote connecting)
evo = mne.read_evokeds(evo_fname)[0]
sens = np.zeros(evo.info['nchan'])
all_chs = mne.pick_types(evo.info, meg=True, eeg=True, exclude=[])
sens[all_chs] = np.sum(fwd['sol']['data'][:, fwd['src'][0]['nuse']+active_verts]*10**-7,axis=1)
evo._data[all_chs] = np.repeat(sens[all_chs].reshape((len(all_chs),1)), repeats=evo._data.shape[1], axis=1)
estimate = mne.minimum_norm.apply_inverse(evo, inverse_operator, 1/9, 'sLORETA', verbose='WARNING')
estimate_cerb = np.linalg.norm(estimate.data[fwd['src'][0]['nuse']:estimate.shape[0], :], axis=1)
cort_data = np.linalg.norm(estimate.data[:fwd['src'][0]['nuse'], :], axis=1)

plot_cerebellum_data(estimate_cerb, fwd['src'], src_whole, cb_data, cort_data=cort_data, flatmap_cmap='bwr',
                         mayavi_cmap='OrRd', smoothing_steps=0, view='flatmap', sub_sampling=cerebellum_subsampling,
                         cmap_lims=[25,75])

In [ ]:
# Sensitivity maps - cerebellum only
for ch_type in ['mag', 'grad', 'eeg']:
    ch_inds = mne.channel_indices_by_type(fwd['info'])
    signal_norms = np.linalg.norm(fwd['sol']['data'][ch_inds[ch_type], fwd['src'][0]['nuse']:], axis=0)
    plot_cerebellum_data(signal_norms, fwd['src'], src_whole, cb_data, cort_data=np.zeros(fwd['src'][0]['nuse']), flatmap_cmap='bwr',
                             mayavi_cmap='OrRd', smoothing_steps=0, view='normal', sub_sampling=cerebellum_subsampling,
                             cmap_lims=[25,75])
# Sensitivity maps - with cortex
for ch_type in ['mag', 'grad', 'eeg']:
    ch_inds = mne.channel_indices_by_type(fwd['info'])
    signal_norms = np.linalg.norm(fwd['sol']['data'][ch_inds[ch_type], :], axis=0)
    plot_cerebellum_data(signal_norms[fwd['src'][0]['nuse']:], fwd['src'], src_whole, cb_data, cort_data=signal_norms[:fwd['src'][0]['nuse']], flatmap_cmap='bwr',
                             mayavi_cmap='OrRd', smoothing_steps=0, view='normal', sub_sampling=cerebellum_subsampling,
                             cmap_lims=[25,75])